# ⚡ Notebook 02 — Event Stream Simulator
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 1 — Streaming & Event Simulator

---

**Objetivo:** Simular um fluxo de dados em tempo real de viagens do Citi Bike, reproduzindo a "pressão logística" que o sistema sofre ao longo do dia. O simulador lê dados históricos da camada Bronze e os emite como micro-lotes (micro-batches), simulando um cenário de streaming.

**Arquitetura:**
```
┌─────────────────────┐     ┌─────────────────────┐     ┌──────────────────┐
│  Bronze/trips       │ ──► │  Event Simulator     │ ──► │  streaming/       │
│  (Parquet histórico)│     │  (Python Generator)  │     │  events/          │
└─────────────────────┘     │  - Micro-batches     │     │  (JSON micro-lotes│
                            │  - Velocidade ajust. │     │   particionados)  │
┌─────────────────────┐     │  - Enriquecimento    │     └──────────────────┘
│  GBFS Station Feed  │ ──► │    com station status│              │
│  (tempo real)       │     └─────────────────────┘              ▼
└─────────────────────┘                              ┌──────────────────┐
                                                     │  Spark Structured│
                                                     │  Streaming       │
                                                     │  (consumidor)    │
                                                     └──────────────────┘
```

**Ferramentas de Big Data:**
- **Spark Structured Streaming** — Consumidor dos micro-lotes
- **PySpark** — Processamento do stream
- **JSON micro-batch** — Formato intermediário do stream

## 0. Setup

In [1]:
!pip install pyspark pyarrow -q

In [2]:
import os
import sys
import json
import time
import shutil
import random
import logging
import threading
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, IntegerType, LongType
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_streaming')


In [3]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / 'dados'
BRONZE_TRIPS = DATA_DIR / 'bronze' / 'trips'
BRONZE_STATIONS = DATA_DIR / 'bronze' / 'stations'

# Diretórios do streaming — usar filesystem Linux (evita problemas de permissão no WSL)
STREAM_DIR = Path.home() / 'citibike' / 'streaming'
STREAM_EVENTS = STREAM_DIR / 'events'
STREAM_CHECKPOINT = STREAM_DIR / 'checkpoint'
STREAM_OUTPUT = STREAM_DIR / 'output'

# Limpar diretórios de streaming (fresh start)
for d in [STREAM_EVENTS, STREAM_CHECKPOINT, STREAM_OUTPUT]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# Parâmetros do simulador
BATCH_SIZE = 50
BATCH_INTERVAL_SEC = 2.0
TOTAL_BATCHES = 100
SIMULATION_DATE = '2024-07-15'
SPEED_FACTOR = 60

print(f"📁 Stream events dir: {STREAM_EVENTS}")
print(f"⚙️  Batch size: {BATCH_SIZE} viagens")
print(f"⏱️  Intervalo: {BATCH_INTERVAL_SEC}s")
print(f"📅  Data simulada: {SIMULATION_DATE}")
print(f"⚡ Speed factor: {SPEED_FACTOR}x")

📁 Stream events dir: /home/joao/citibike/streaming/events
⚙️  Batch size: 50 viagens
⏱️  Intervalo: 2.0s
📅  Data simulada: 2024-07-15
⚡ Speed factor: 60x


In [4]:
# Inicializar Spark
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("CitiBike_StreamSimulator")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} inicializado")
print(f"   Cores: {spark.sparkContext.defaultParallelism}")
print(f"   Driver memory: {spark.conf.get('spark.driver.memory')}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 23:44:04 WARN Utils: Your hostname, DESKTOP-K9JUL83, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/29 23:44:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 23:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.1 inicializado
   Cores: 12
   Driver memory: 4g
